[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/097.%20The%20Last-Mile%20Delivery%20%26%20Micro-Fulfillment%20Problem/P97-Tier-4_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 97. The Last-Mile Delivery & Micro-Fulfillment Problem
## Tier 4: The AI/ML/RL Augmentation Method (Neural Architecture Search)

### Key assumptions
- Neural architectures can be automatically discovered for routing problems
- Different encoder-decoder combinations suit different problem characteristics
- Attention mechanisms can capture complex spatial relationships
- Graph neural networks excel at route optimization tasks
- Performance can be evaluated through training and validation cycles

### Approach (step-by-step)
1. **Architecture Space Definition**: Define search space with encoders, attention mechanisms, and decoders
2. **Controller Network**: Train RNN controller to sample promising architectures
3. **Architecture Evaluation**: Train and evaluate each sampled architecture
4. **Performance Feedback**: Update controller based on architecture performance
5. **Convergence**: Iterate to discover optimal neural architecture

### What to look for in the results
- Best discovered architecture configuration
 - Performance improvement over baseline architectures
- Search efficiency and convergence behavior
- Generalization across different problem instances

### Concrete example (from the source)
NAS framework discovers optimal architecture after 100 search iterations with 15.7% improvement over baseline transformer architecture.

In [1]:
from typing import Tuple, List, Dict, Set
from scipy import optimize
from itertools import product

# Import required libraries for Neural Architecture Search
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional
import random
import warnings
warnings.filterwarnings('ignore')

# Set style for visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class Customer:
    """Represents a customer location with demand"""
    id: int
    x: float
    y: float
    demand: int
    time_window: Tuple[int, int]  # (start, end) hours

@dataclass
class Architecture:
    """Represents a neural architecture configuration"""
    encoder_type: str  # 'transformer', 'gcn', 'cnn'
    attention_heads: int  # 1, 2, 4, or 8
    decoder_type: str  # 'pointer', 'reinforcement', 'constructive'
    embedding_dim: int  # 64, 128, or 256
    hidden_layers: int  # 2, 3, or 4
    performance: float = 0.0  # Validation performance
    training_time: float = 0.0  # Training time in seconds

class NeuralArchitectureSearch:
    """Neural Architecture Search for last-mile delivery routing"""
    
    def __init__(self, max_iterations: int = 100, population_size: int = 10):
        self.max_iterations = max_iterations
        self.population_size = population_size
        
        # Architecture search space
        self.encoder_types = ['transformer', 'gcn', 'cnn']
        self.attention_heads = [1, 2, 4, 8]
        self.decoder_types = ['pointer', 'reinforcement', 'constructive']
        self.embedding_dims = [64, 128, 256]
        self.hidden_layers = [2, 3, 4]
        
        # Search history
        self.search_history = []
        self.best_architecture = None
        self.best_performance = float('-inf')
        
        # Performance tracking
        self.performance_history = []
        self.diversity_history = []
        
    def sample_architecture(self) -> Architecture:
        """Sample a random architecture from the search space"""
        return Architecture(
            encoder_type=random.choice(self.encoder_types),
            attention_heads=random.choice(self.attention_heads),
            decoder_type=random.choice(self.decoder_types),
            embedding_dim=random.choice(self.embedding_dims),
            hidden_layers=random.choice(self.hidden_layers)
        )
    
    def calculate_architecture_complexity(self, arch: Architecture) -> float:
        """Calculate complexity score for an architecture"""
        complexity = 0.0
        
        # Encoder complexity
        if arch.encoder_type == 'transformer':
            complexity += 1.0
        elif arch.encoder_type == 'gcn':
            complexity += 0.8
        else:  # cnn
            complexity += 0.6
        
        # Attention complexity
        complexity += arch.attention_heads * 0.1
        
        # Decoder complexity
        if arch.decoder_type == 'pointer':
            complexity += 0.9
        elif arch.decoder_type == 'reinforcement':
            complexity += 1.0
        else:  # constructive
            complexity += 0.7
        
        # Embedding and layer complexity
        complexity += arch.embedding_dim / 128 * 0.5
        complexity += arch.hidden_layers * 0.2
        
        return complexity
    
    def simulate_training(self, arch: Architecture, customers: List[Customer]) -> Tuple[float, float]:
        """Simulate training and evaluation of an architecture"""
        # Simulate training time based on complexity
        complexity = self.calculate_architecture_complexity(arch)
        training_time = complexity * random.uniform(5, 15)  # 5-15 seconds scaled by complexity
        
        # Simulate performance based on architecture characteristics
        base_performance = 0.7  # Base performance
        
        # Encoder performance contribution
        if arch.encoder_type == 'gcn':
            base_performance += 0.12  # GCN is good for routing
        elif arch.encoder_type == 'transformer':
            base_performance += 0.08
        else:  # cnn
            base_performance += 0.04
        
        # Attention heads contribution (optimal at 4)
        if arch.attention_heads == 4:
            base_performance += 0.10
        elif arch.attention_heads == 2:
            base_performance += 0.06
        elif arch.attention_heads == 8:
            base_performance += 0.02
        
        # Decoder contribution
        if arch.decoder_type == 'pointer':
            base_performance += 0.08  # Pointer networks are good for sequences
        elif arch.decoder_type == 'reinforcement':
            base_performance += 0.06
        else:  # constructive
            base_performance += 0.04
        
        # Embedding dimension contribution
        if arch.embedding_dim == 128:
            base_performance += 0.06
        elif arch.embedding_dim == 256:
            base_performance += 0.04
        else:  # 64
            base_performance += 0.02
        
        # Hidden layers contribution (optimal at 3)
        if arch.hidden_layers == 3:
            base_performance += 0.04
        elif arch.hidden_layers == 2:
            base_performance += 0.02
        
        # Add noise and problem-specific factors
        problem_factor = min(len(customers) / 20, 1.0)  # Scale with problem size
        noise = random.gauss(0, 0.02)  # Random noise
        
        final_performance = base_performance + problem_factor * 0.1 + noise
        final_performance = max(0.0, min(1.0, final_performance))  # Clamp to [0, 1]
        
        return final_performance, training_time
    
    def evaluate_architecture(self, arch: Architecture, customers: List[Customer]) -> float:
        """Evaluate architecture performance"""
        performance, training_time = self.simulate_training(arch, customers)
        arch.performance = performance
        arch.training_time = training_time
        return performance
    
    def calculate_diversity(self, architectures: List[Architecture]) -> float:
        """Calculate diversity in current architecture population"""
        if len(architectures) <= 1:
            return 0.0
        
        diversity = 0.0
        total_pairs = 0
        
        for i in range(len(architectures)):
            for j in range(i + 1, len(architectures)):
                arch1, arch2 = architectures[i], architectures[j]
                
                # Calculate architectural differences
                diff = 0
                if arch1.encoder_type != arch2.encoder_type:
                    diff += 1
                if arch1.attention_heads != arch2.attention_heads:
                    diff += 1
                if arch1.decoder_type != arch2.decoder_type:
                    diff += 1
                if arch1.embedding_dim != arch2.embedding_dim:
                    diff += 1
                if arch1.hidden_layers != arch2.hidden_layers:
                    diff += 1
                
                diversity += diff / 5  # Normalize by max possible differences
                total_pairs += 1
        
        return diversity / total_pairs if total_pairs > 0 else 0.0
    
    def mutate_architecture(self, arch: Architecture) -> Architecture:
        """Create mutated version of an architecture"""
        new_arch = Architecture(
            encoder_type=arch.encoder_type,
            attention_heads=arch.attention_heads,
            decoder_type=arch.decoder_type,
            embedding_dim=arch.embedding_dim,
            hidden_layers=arch.hidden_layers
        )
        
        # Randomly mutate one component
        mutation_type = random.choice(['encoder', 'attention', 'decoder', 'embedding', 'layers'])
        
        if mutation_type == 'encoder':
            new_arch.encoder_type = random.choice(self.encoder_types)
        elif mutation_type == 'attention':
            new_arch.attention_heads = random.choice(self.attention_heads)
        elif mutation_type == 'decoder':
            new_arch.decoder_type = random.choice(self.decoder_types)
        elif mutation_type == 'embedding':
            new_arch.embedding_dim = random.choice(self.embedding_dims)
        else:  # layers
            new_arch.hidden_layers = random.choice(self.hidden_layers)
        
        return new_arch
    
    def search(self, customers: List[Customer]) -> Architecture:
        """Run Neural Architecture Search"""
        print("Starting Neural Architecture Search...")
        
        # Initialize population
        population = [self.sample_architecture() for _ in range(self.population_size)]
        
        # Evaluate initial population
        for arch in population:
            self.evaluate_architecture(arch, customers)
        
        # Sort by performance
        population.sort(key=lambda x: x.performance, reverse=True)
        self.best_architecture = population[0]
        self.best_performance = population[0].performance
        
        print(f"Initial best performance: {self.best_performance:.3f}")
        
        # Search iterations
        for iteration in range(self.max_iterations):
            # Create new population through selection and mutation
            new_population = []
            
            # Keep top performers (elitism)
            elite_size = max(1, self.population_size // 4)
            new_population.extend(population[:elite_size])
            
            # Generate new architectures through mutation
            while len(new_population) < self.population_size:
                # Select parent from top performers
                parent = random.choice(population[:self.population_size // 2])
                child = self.mutate_architecture(parent)
                self.evaluate_architecture(child, customers)
                new_population.append(child)
            
            # Update population
            population = sorted(new_population, key=lambda x: x.performance, reverse=True)
            
            # Update best solution
            if population[0].performance > self.best_performance:
                self.best_architecture = population[0]
                self.best_performance = population[0].performance
            
            # Record metrics
            self.performance_history.append(population[0].performance)
            self.diversity_history.append(self.calculate_diversity(population))
            
            # Progress reporting
            if (iteration + 1) % 20 == 0:
                print(f"Iteration {iteration + 1}/{self.max_iterations}: "
                      f"Best performance = {self.best_performance:.3f}")
        
        print(f"\nNAS completed!")
        print(f"Final best performance: {self.best_performance:.3f}")
        
        return self.best_architecture

print("Neural Architecture Search implementation complete")

Neural Architecture Search implementation complete


In [ ]:
# Initialize the concrete example from the problem statement
# Create a representative delivery problem
customers_data = [
    (0, 10, 15, 2, (8, 12)),   # Customer 0
    (1, 20, 25, 3, (9, 11)),   # Customer 1
    (2, 15, 35, 1, (10, 14)),  # Customer 2
    (3, 30, 20, 4, (8, 10)),   # Customer 3
    (4, 25, 30, 2, (11, 15)),  # Customer 4
    (5, 35, 40, 3, (12, 16)),  # Customer 5
    (6, 40, 15, 1, (9, 13)),   # Customer 6
    (7, 45, 35, 2, (10, 14)),  # Customer 7
    (8, 50, 25, 3, (13, 17)),  # Customer 8
    (9, 55, 45, 2, (8, 12)),   # Customer 9
    (10, 60, 30, 1, (11, 15)), # Customer 10
    (11, 65, 40, 4, (9, 13)),  # Customer 11
]

# Create Customer objects
customers = [Customer(cust_id, x, y, demand, time_window) 
             for cust_id, x, y, demand, time_window in customers_data]

print(f"Initialized {len(customers)} customers for NAS evaluation")
print("\nCustomer summary:")
for customer in customers[:5]:  # Show first 5
    print(f"Customer {customer.id}: ({customer.x}, {customer.y}), demand={customer.demand}, "
          f"window={customer.time_window}")
print(f"... and {len(customers) - 5} more customers")

Initialized 12 customers for NAS evaluation

Customer summary:
Customer 0: (10, 15), demand=2, window=(8, 12)
Customer 1: (20, 25), demand=3, window=(9, 11)
Customer 2: (15, 35), demand=1, window=(10, 14)
Customer 3: (30, 20), demand=4, window=(8, 10)
Customer 4: (25, 30), demand=2, window=(11, 15)
... and 7 more customers


In [ ]:
# Run Neural Architecture Search
# Parameters from the problem statement
max_iterations = 100
population_size = 10

# Create NAS instance
nas = NeuralArchitectureSearch(
    max_iterations=max_iterations,
    population_size=population_size
)

# Run architecture search
best_architecture = nas.search(customers)

print(f"\nBest discovered architecture:")
print(f"  Encoder: {best_architecture.encoder_type}")
print(f"  Attention heads: {best_architecture.attention_heads}")
print(f"  Decoder: {best_architecture.decoder_type}")
print(f"  Embedding dim: {best_architecture.embedding_dim}")
print(f"  Hidden layers: {best_architecture.hidden_layers}")
print(f"  Performance: {best_architecture.performance:.3f}")
print(f"  Training time: {best_architecture.training_time:.1f}s")

Starting Neural Architecture Search...
Initial best performance: 1.000
Iteration 20/100: Best performance = 1.000
Iteration 40/100: Best performance = 1.000
Iteration 60/100: Best performance = 1.000
Iteration 80/100: Best performance = 1.000
Iteration 100/100: Best performance = 1.000

NAS completed!
Final best performance: 1.000

Best discovered architecture:
  Encoder: gcn
  Attention heads: 1
  Decoder: pointer
  Embedding dim: 256
  Hidden layers: 3
  Performance: 1.000
  Training time: 39.0s


In [ ]:
# Analyze NAS results
print("=" * 60)
print("NEURAL ARCHITECTURE SEARCH RESULTS")
print("=" * 60)

# Calculate performance metrics
initial_performance = nas.performance_history[0] if nas.performance_history else 0
final_performance = nas.best_performance
improvement = (final_performance - initial_performance) / initial_performance * 100 if initial_performance > 0 else 0

print(f"🎯 SEARCH PERFORMANCE:")
print(f"   Initial best performance: {initial_performance:.3f}")
print(f"   Final best performance: {final_performance:.3f}")
print(f"   Total improvement: {improvement:.1f}%")
print(f"   Search efficiency: {improvement/max_iterations:.2f}% per iteration")

# Architecture analysis
print(f"\n🏗️ BEST ARCHITECTURE ANALYSIS:")
print(f"   Configuration: {best_architecture.encoder_type}-{best_architecture.attention_heads}h-"
      f"{best_architecture.decoder_type}-{best_architecture.embedding_dim}d-{best_architecture.hidden_layers}l")
print(f"   Complexity score: {nas.calculate_architecture_complexity(best_architecture):.2f}")
print(f"   Training time: {best_architecture.training_time:.1f} seconds")

# Diversity analysis
if nas.diversity_history:
    initial_diversity = nas.diversity_history[0]
    final_diversity = nas.diversity_history[-1]
    avg_diversity = np.mean(nas.diversity_history)
    
    print(f"\n🔄 DIVERSITY ANALYSIS:")
    print(f"   Initial diversity: {initial_diversity:.3f}")
    print(f"   Final diversity: {final_diversity:.3f}")
    print(f"   Average diversity: {avg_diversity:.3f}")
    
    if final_diversity < initial_diversity * 0.5:
        diversity_status = "Converged"
    elif final_diversity < initial_diversity * 0.8:
        diversity_status = "Partially converged"
    else:
        diversity_status = "Diverse"
    
    print(f"   Diversity status: {diversity_status}")

# Search space coverage
print(f"\n📊 SEARCH SPACE ANALYSIS:")
total_possible = (len(nas.encoder_types) * len(nas.attention_heads) * 
                 len(nas.decoder_types) * len(nas.embedding_dims) * len(nas.hidden_layers))
print(f"   Total search space size: {total_possible:,} architectures")
print(f"   Architectures evaluated: {max_iterations * population_size}")
print(f"   Search coverage: {(max_iterations * population_size) / total_possible * 100:.2f}%")
print(f"   Search efficiency: {'High' if improvement > 15 else 'Medium' if improvement > 10 else 'Low'}")

NEURAL ARCHITECTURE SEARCH RESULTS
🎯 SEARCH PERFORMANCE:
   Initial best performance: 1.000
   Final best performance: 1.000
   Total improvement: 0.0%
   Search efficiency: 0.00% per iteration

🏗️ BEST ARCHITECTURE ANALYSIS:
   Configuration: gcn-1h-pointer-256d-3l
   Complexity score: 3.40
   Training time: 39.0 seconds

🔄 DIVERSITY ANALYSIS:
   Initial diversity: 0.498
   Final diversity: 0.311
   Average diversity: 0.411
   Diversity status: Partially converged

📊 SEARCH SPACE ANALYSIS:
   Total search space size: 324 architectures
   Architectures evaluated: 1000
   Search coverage: 308.64%
   Search efficiency: Low


In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Neural Architecture Search - Last-Mile Delivery', fontsize=16, fontweight='bold')

# 1. Performance convergence
ax1 = axes[0, 0]
iterations = range(1, len(nas.performance_history) + 1)
ax1.plot(iterations, nas.performance_history, 'b-', linewidth=2, marker='o', markersize=3)
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Performance Score')
ax1.set_title('NAS Performance Convergence')
ax1.grid(True, alpha=0.3)
ax1.fill_between(iterations, nas.performance_history, alpha=0.3)

# Add convergence line
if len(nas.performance_history) > 10:
    # Fit exponential convergence curve
    from scipy.optimize import curve_fit
    
    def exp_func(x, a, b, c):
        return a * np.exp(-b * x) + c
    
    try:
        popt, _ = curve_fit(exp_func, iterations, nas.performance_history, 
                           p0=[0.1, 0.05, max(nas.performance_history)], maxfev=1000)
        smooth_curve = exp_func(np.array(iterations), *popt)
        ax1.plot(iterations, smooth_curve, 'r--', alpha=0.7, label='Convergence trend')
        ax1.legend()
    except:
        pass

# 2. Architecture component distribution
ax2 = axes[0, 1]

# Analyze component distribution in best architectures
component_analysis = {
    'Encoder': {},
    'Attention': {},
    'Decoder': {},
    'Embedding': {},
    'Layers': {}
}

# Simulate analysis of top performing architectures
top_architectures = []
for _ in range(20):  # Sample 20 good architectures
    arch = nas.sample_architecture()
    # Bias toward better performing configurations
    if arch.encoder_type == 'gcn':
        arch.performance = random.uniform(0.8, 0.95)
    elif arch.encoder_type == 'transformer':
        arch.performance = random.uniform(0.75, 0.9)
    else:
        arch.performance = random.uniform(0.7, 0.85)
    top_architectures.append(arch)

# Count component frequencies
for arch in top_architectures:
    component_analysis['Encoder'][arch.encoder_type] = component_analysis['Encoder'].get(arch.encoder_type, 0) + 1
    component_analysis['Attention'][arch.attention_heads] = component_analysis['Attention'].get(arch.attention_heads, 0) + 1
    component_analysis['Decoder'][arch.decoder_type] = component_analysis['Decoder'].get(arch.decoder_type, 0) + 1
    component_analysis['Embedding'][arch.embedding_dim] = component_analysis['Embedding'].get(arch.embedding_dim, 0) + 1
    component_analysis['Layers'][arch.hidden_layers] = component_analysis['Layers'].get(arch.hidden_layers, 0) + 1

# Create subplots for each component
categories = list(component_analysis.keys())
for i, (category, counts) in enumerate(component_analysis.items()):
    if len(counts) > 0:
        items = list(counts.keys())
        values = list(counts.values())
        
        # Create small subplot within ax2
        y_pos = 0.8 - i * 0.15
        ax2.barh(y_pos, max(values), height=0.1, alpha=0.3, color='gray')
        
        # Add bars for each item
        x_pos = 0
        for item, value in zip(items, values):
            ax2.barh(y_pos, value, height=0.08, left=x_pos, alpha=0.7, 
                   label=f'{item}' if category == 'Encoder' else '')
            ax2.text(x_pos + value/2, y_pos, str(item), ha='center', va='center', fontsize=8)
            x_pos += value
        
        ax2.text(-0.05, y_pos, category, ha='right', va='center', fontweight='bold')

ax2.set_xlim(-0.1, 20)
ax2.set_ylim(-0.1, 1.0)
ax2.set_title('Architecture Component Distribution')
ax2.set_xticks([])
ax2.set_yticks([])
ax2.legend(loc='upper right')

# 3. Diversity over time
ax3 = axes[1, 0]
if nas.diversity_history:
    iterations = range(1, len(nas.diversity_history) + 1)
    ax3.plot(iterations, nas.diversity_history, 'g-', linewidth=2, marker='s', markersize=3)
    ax3.set_xlabel('Iteration')
    ax3.set_ylabel('Diversity Score')
    ax3.set_title('Population Diversity Over Time')
    ax3.grid(True, alpha=0.3)
    ax3.fill_between(iterations, nas.diversity_history, alpha=0.3, color='green')

# 4. Performance comparison with baseline
ax4 = axes[1, 1]

# Compare with baseline architectures
baseline_architectures = {
    'Baseline Transformer': Architecture('transformer', 8, 'pointer', 256, 4),
    'Simple CNN': Architecture('cnn', 1, 'constructive', 64, 2),
    'Basic GCN': Architecture('gcn', 2, 'reinforcement', 128, 3),
    'NAS Best': best_architecture
}

# Evaluate baseline architectures
baseline_performances = []
baseline_names = []
baseline_colors = []

for name, arch in baseline_architectures.items():
    if name == 'NAS Best':
        performance = best_architecture.performance
        color = 'green'
    else:
        performance, _ = nas.simulate_training(arch, customers)
        color = 'blue'
    
    baseline_performances.append(performance)
    baseline_names.append(name)
    baseline_colors.append(color)

bars = ax4.bar(baseline_names, baseline_performances, color=baseline_colors, alpha=0.7)
ax4.set_ylabel('Performance Score')
ax4.set_title('Performance Comparison with Baselines')
ax4.set_ylim(0, 1)

# Add value labels on bars
for bar, perf in zip(bars, baseline_performances):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{perf:.3f}', ha='center', va='bottom', fontweight='bold')

# Add improvement annotation
if len(baseline_performances) >= 3:
    baseline_avg = np.mean(baseline_performances[:-1])  # Exclude NAS Best
    nas_improvement = (baseline_performances[-1] - baseline_avg) / baseline_avg * 100
    
    ax4.text(1.5, 0.9, f'NAS Improvement: {nas_improvement:.1f}%', 
             ha='center', va='center', fontsize=12, fontweight='bold',
             bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))

plt.tight_layout()
plt.show()

print("Neural Architecture Search visualization complete!")

  Classical time: 12.027s
  Quantum time: 2.049s
  Speedup: 5.9x
  Quality improvement: 1.0%

Testing problem size: 100 variables
Market shaping visualization complete!


In [ ]:
# Detailed analysis and recommendations
print("=" * 60)
print('ARCHITECTURE ANALYSIS AND RECOMMENDATIONS')
print("=" * 60)

# Architecture characteristics analysis
print(f"🔍 BEST ARCHITECTURE CHARACTERISTICS:")
print(f"   Encoder Type: {best_architecture.encoder_type}")
if best_architecture.encoder_type == 'gcn':
    print(f"     → Graph Convolutional Networks excel at capturing spatial relationships")
    print(f"     → Ideal for routing problems with customer location graphs")
elif best_architecture.encoder_type == 'transformer':
    print(f"     → Transformers handle sequential dependencies well")
    print(f"     → Good for time-windowed delivery problems")
else:
    print(f"     → CNNs capture local spatial patterns efficiently")
    print(f"     → Suitable for grid-based city layouts")

print(f"   Attention Heads: {best_architecture.attention_heads}")
if best_architecture.attention_heads == 4:
    print(f"     → Optimal balance between expressiveness and computational efficiency")
elif best_architecture.attention_heads > 4:
    print(f"     → High expressiveness but increased computational cost")
else:
    print(f"     → Computationally efficient but limited expressiveness")

print(f"   Decoder Type: {best_architecture.decoder_type}")
if best_architecture.decoder_type == 'pointer':
    print(f"     → Pointer networks naturally handle sequence generation")
    print(f"     → Perfect for route construction problems")
elif best_architecture.decoder_type == 'reinforcement':
    print(f"     → RL-based decoders learn optimal policies")
    print(f"     → Adaptable to dynamic routing environments")
else:
    print(f"     → Constructive decoders build solutions step-by-step")
    print(f"     → Interpretable and stable performance")

# Performance analysis
print(f"\n📈 PERFORMANCE METRICS:")
print(f"   Final performance score: {final_performance:.3f}")
print(f"   Improvement over initial: {improvement:.1f}%")
print(f"   Training efficiency: {final_performance/best_architecture.training_time:.3f} performance/second")

# Search efficiency
convergence_iteration = None
for i, perf in enumerate(nas.performance_history):
    if perf >= final_performance * 0.95:  # 95% of final performance
        convergence_iteration = i + 1
        break

if convergence_iteration:
    print(f"   Convergence iteration: {convergence_iteration} (95% of final performance)")
    print(f"   Search efficiency: {'High' if convergence_iteration < 50 else 'Medium' if convergence_iteration < 80 else 'Low'}")

# Recommendations
print(f"\n💡 RECOMMENDATIONS:")

if final_performance > 0.85:
    print(f"   ✅ Architecture shows excellent performance")
    print(f"   ✅ Ready for production deployment")
elif final_performance > 0.75:
    print(f"   ⚠️ Architecture shows good performance")
    print(f"   📝 Consider hyperparameter tuning for improvement")
else:
    print(f"   ❌ Architecture performance needs improvement")
    print(f"   🔄 Consider extended search or different search strategy")

if best_architecture.training_time < 10:
    print(f"   ⚡ Fast training enables rapid prototyping")
elif best_architecture.training_time < 20:
    print(f"   ⏱️ Moderate training time acceptable for most applications")
else:
    print(f"   🐌 Long training time may limit real-time applicability")

# Generalization assessment
print(f"\n🌐 GENERALIZATION ASSESSMENT:")
problem_complexity = len(customers)
if problem_complexity <= 10:
    print(f"   Tested on small-scale problem ({problem_complexity} customers)")
    print(f"   🔄 Should validate on larger instances for generalization")
elif problem_complexity <= 20:
    print(f"   Tested on medium-scale problem ({problem_complexity} customers)")
    print(f"   ✅ Good balance for testing and generalization")
else:
    print(f"   Tested on large-scale problem ({problem_complexity} customers)")
    print(f"   ✅ Likely to generalize well to similar instances")

print(f"\n🎯 FINAL ASSESSMENT:")
if improvement > 15 and final_performance > 0.8:
    assessment = "Excellent - NAS discovered high-performing architecture"
elif improvement > 10 and final_performance > 0.7:
    assessment = "Good - NAS found effective architecture"
elif improvement > 5:
    assessment = "Moderate - NAS shows some improvement"
else:
    assessment = "Limited - NAS needs refinement"

print(f"   {assessment}")

ARCHITECTURE ANALYSIS AND RECOMMENDATIONS
🔍 BEST ARCHITECTURE CHARACTERISTICS:
   Encoder Type: gcn
     → Graph Convolutional Networks excel at capturing spatial relationships
     → Ideal for routing problems with customer location graphs
   Attention Heads: 1
     → Computationally efficient but limited expressiveness
   Decoder Type: pointer
     → Pointer networks naturally handle sequence generation
     → Perfect for route construction problems

📈 PERFORMANCE METRICS:
   Final performance score: 1.000
   Improvement over initial: 0.0%
   Training efficiency: 0.026 performance/second
   Convergence iteration: 1 (95% of final performance)
   Search efficiency: High

💡 RECOMMENDATIONS:
   ✅ Architecture shows excellent performance
   ✅ Ready for production deployment
   🐌 Long training time may limit real-time applicability

🌐 GENERALIZATION ASSESSMENT:
   Tested on medium-scale problem (12 customers)
   ✅ Good balance for testing and generalization

🎯 FINAL ASSESSMENT:
   Limited 

### Why this Tier exists vs Tiers 1-3
Neural Architecture Search provides advanced AI capabilities beyond traditional optimization:
- **Automated design**: Discovers optimal architectures vs manual design
- **Adaptability**: Learns architectures suited to specific problem characteristics
- **Performance optimization**: Systematically explores architecture space for best performance
- **Generalization**: Found architectures often generalize better to new instances

### Pros / Cons vs Tiers 1-3
**Pros:**
- Automatically discovers optimal neural architectures
- Adapts to specific problem characteristics
- Can outperform manually designed architectures
- Systematic exploration of design space
- Found architectures often generalize well
- Reduces need for manual architecture engineering

**Cons:**
- Computationally expensive search process
- Requires significant computational resources
- Search space can be very large
- No guarantee of finding globally optimal architecture
- Performance depends on search strategy quality
- May require domain expertise for search space design

### When to use this Tier
- When neural network performance is critical
- For problems where manual architecture design is difficult
- When sufficient computational resources are available
- For production systems where performance justifies search cost
- When developing reusable AI solutions for routing
- For research and development of advanced AI methods